In [0]:
spark.sql("USE CATALOG fraud_detection")
spark.sql("USE SCHEMA source")

In [0]:
%pip install faker
from faker import Faker
import random
import pandas as pd
from datetime import datetime

fake = Faker()

def generate_transactions(batch_size=50):
    data = []
    for _ in range(batch_size):
        data.append({
            "transaction_id": fake.uuid4(),
            "card_id": fake.credit_card_number(),
            "amount": round(random.uniform(100, 80000), 2),
            "location": fake.city(),
            "timestamp": datetime.now()
        })
    return pd.DataFrame(data)

pdf = generate_transactions(50)
sdf = spark.createDataFrame(pdf)

sdf.write \
  .format("delta") \
  .mode("append") \
  .saveAsTable("fraud_detection.source.transactions")

print("Batch generated successfully!")

In [0]:
%sql
select count(*) from fraud_detection.source.transactions;